In [1]:
import base64
import struct

# Measurement and scaling constants
MAX_STANDARD_DEVIATION: float = 1.0
MAX_RELATIVE_HEIGHT: float = 20.0

# Temperature scaling range (Celsius)
MIN_TEMPERATURE_C: float = -30.0
MAX_TEMPERATURE_C: float = 80.0

# Battery voltage range (millivolts)
MIN_BATTERY_MV: int = 2000
MAX_BATTERY_MV: int = 5000


def map_u8_to_float(byte_val: int, min_val: float, max_val: float) -> float:
    if byte_val <= 0:
        return min_val
    if byte_val >= 255:
        return max_val
    return min_val + (byte_val / 255.0) * (max_val - min_val)

def map_u16_to_float(u16_val: int, min_val: float, max_val: float) -> float:
    if u16_val <= 0:
        return min_val
    if u16_val >= 65535:
        return max_val
    return min_val + (u16_val / 65535.0) * (max_val - min_val)

def map_u8_to_int(byte_val: int, min_val: int, max_val: int) -> int:
    if byte_val <= 0:
        return min_val
    if byte_val >= 255:
        return max_val
    scaled = byte_val / 255.0
    return int(round(min_val + scaled * (max_val - min_val)))


# Example base64 packet
packet_b64 = "Y2HrOsAWxWc+BwoWxWc+Bwq93RSbAgk="

# Decode Base64
packet_bytes = base64.b64decode(packet_b64)
print(f"Decoded {len(packet_bytes)} bytes: {packet_bytes.hex()}")

# Define formats
HEADER_FORMAT = "<BBBBB"           # battery, temp, lat, lon, charge_state_fraction (5 bytes)
MEASUREMENT_FORMAT = "<BHBBB"     # u8, u16 (LE), u8, u8, u8 (6 bytes)
HEADER_SIZE = struct.calcsize(HEADER_FORMAT)
MEASUREMENT_SIZE = struct.calcsize(MEASUREMENT_FORMAT)

# Parse header
header_data = packet_bytes[:HEADER_SIZE]
battery_raw, temp_raw, lat, lon, charge_state_fraction = struct.unpack(HEADER_FORMAT, header_data)

battery = map_u8_to_int(battery_raw, MIN_BATTERY_MV, MAX_BATTERY_MV)
temp = map_u8_to_float(temp_raw, MIN_TEMPERATURE_C, MAX_TEMPERATURE_C)

# Extract 4 values (2 bits each)
values = [
    (charge_state_fraction >> 6) & 0b11,  # Completed
    (charge_state_fraction >> 4) & 0b11,  # Charging
    (charge_state_fraction >> 2) & 0b11,  # Recoverable
    (charge_state_fraction >> 0) & 0b11   # Nonrecoverable
]

names = ["Completed", "Charging", "Recoverable", "Nonrecoverable"]

# Compute percentages
total = sum(values)
percentages = [v / total * 100 if total else 0 for v in values]

print("\nHeader:")
print(f"  battery: {battery}")
print(f"  temp:    {temp}")
print(f"  lat:     {lat}")
print(f"  lon:     {lon}")

# Print results
print(f"  Charge State Fraction (u8): {charge_state_fraction:#010b} ({charge_state_fraction})\n")
for name, value, pct in zip(names, values, percentages):
    print(f"{name:15s}: value = {value}, percentage = {pct:5.2f}%")

# 4️⃣ Parse measurements
measurements_bytes = packet_bytes[HEADER_SIZE:]
measurements = []

for i in range(0, len(measurements_bytes), MEASUREMENT_SIZE):
    chunk = measurements_bytes[i:i+MEASUREMENT_SIZE]
    if len(chunk) < MEASUREMENT_SIZE:
        break  # stop if incomplete
    uid, raw_rel_mean, raw_rel_std, num_used, num_seen = struct.unpack(MEASUREMENT_FORMAT, chunk)
    rel_mean = map_u16_to_float(raw_rel_mean, 0, MAX_RELATIVE_HEIGHT)
    rel_std = map_u8_to_float(raw_rel_std, 0, MAX_STANDARD_DEVIATION)

    measurements.append({
        "uid": uid,
        "relative_height_mean": rel_mean,
        "relative_height_std": rel_std,
        "num_observations_used": num_used,
        "num_observations_seen": num_seen
    })

# 5️⃣ Display results
print(f"\nParsed {len(measurements)} measurement(s):")
for m in measurements:
    print(m)

## test 4


## test 3
# Header:
#   battery: 3188
#   temp:    17.01960784313725
#   lat:     205
#   lon:     61

# Parsed 3 measurement(s):
# {'uid': 50, 'relative_height_mean': 4.895094224460212, 'relative_height_std': 0.30196078431372547, 'num_observations_used': 13, 'num_observations_seen': 27}
# {'uid': 111, 'relative_height_mean': 4.124818799114976, 'relative_height_std': 1.0, 'num_observations_used': 8, 'num_observations_seen': 8}
# {'uid': 110, 'relative_height_mean': 4.833142595559624, 'relative_height_std': 1.0, 'num_observations_used': 9, 'num_observations_seen': 9}

## test 2
# Header:
#   battery: 3224
#   temp:    20.470588235294116
#   lat:     214
#   lon:     66

# Parsed 3 measurement(s):
# {'uid': 50, 'relative_height_mean': 3.930724040588998, 'relative_height_std': 0.2549019607843137, 'num_observations_used': 13, 'num_observations_seen': 38}
# {'uid': 111, 'relative_height_mean': 4.124818799114976, 'relative_height_std': 1.0, 'num_observations_used': 8, 'num_observations_seen': 8}
# {'uid': 110, 'relative_height_mean': 4.833142595559624, 'relative_height_std': 1.0, 'num_observations_used': 9, 'num_observations_seen': 9}

## test 1
# Header:
#   battery: 3212
#   temp:    16.15686274509804
#   lat:     0
#   lon:     0

# Parsed 3 measurement(s):
# {'uid': 153, 'relative_height_mean': 2.7768368047608147, 'relative_height_std': 0.3333333333333333, 'num_observations_used': 13, 'num_observations_seen': 53}
# {'uid': 0, 'relative_height_mean': 0, 'relative_height_std': 0, 'num_observations_used': 128, 'num_observations_seen': 255}
# {'uid': 0, 'relative_height_mean': 0, 'relative_height_std': 0, 'num_observations_used': 128, 'num_observations_seen': 255}


Decoded 23 bytes: 6361eb3ac016c5673e070a16c5673e070abddd149b0209

Header:
  battery: 3165
  temp:    11.843137254901961
  lat:     235
  lon:     58
  Charge State Fraction (u8): 0b11000000 (192)

Completed      : value = 0, percentage =  0.00%
Charging       : value = 0, percentage =  0.00%
Recoverable    : value = 0, percentage =  0.00%
Nonrecoverable : value = 3, percentage = 100.00%

Parsed 3 measurement(s):
{'uid': 22, 'relative_height_mean': 8.107118333714809, 'relative_height_std': 0.24313725490196078, 'num_observations_used': 7, 'num_observations_seen': 10}
{'uid': 22, 'relative_height_mean': 8.107118333714809, 'relative_height_std': 0.24313725490196078, 'num_observations_used': 7, 'num_observations_seen': 10}
{'uid': 189, 'relative_height_mean': 1.6299687190051118, 'relative_height_std': 0.6078431372549019, 'num_observations_used': 2, 'num_observations_seen': 9}


    const PACKET_SIZE: usize = PACKET_HEADER_SIZE + (MAX_MEASUREMENT_PACKETS * MEASUREMENT_PACKET_SIZE);
    
    measurement

    pub fn to_bytes(&self) -> [u8; MEASUREMENT_PACKET_SIZE] {
        let mut data = [0u8; MEASUREMENT_PACKET_SIZE];
        data[0] = self.uid;
        data[1..3].copy_from_slice(&self.relative_height_mean.to_le_bytes());
        data[3] = self.relative_height_std;
        data[4] = self.num_observations_used;
        data[5] = self.num_observations_seen;
        data
    }

    packet:

        pub fn to_bytes(&self) -> Vec<u8, PACKET_SIZE> {
        let mut data = Vec::<u8, PACKET_SIZE>::new();
        data.extend_from_slice(&self.battery.to_le_bytes()).ok();
        data.extend_from_slice(&self.temp.to_le_bytes()).ok();
        data.extend_from_slice(&self.lat.to_le_bytes()).ok();
        data.extend_from_slice(&self.lon.to_le_bytes()).ok();
        for measurement in self.measurements.iter() {
            let meas_bytes = measurement.to_bytes();
            data.extend_from_slice(&meas_bytes).ok();
        }
        data
    }


